# Instacart Project Machine Learning Feature Engineering #

In [1]:
import numpy as np
import pandas as pd

In [49]:
# load order_product_prior into dataframe to studies useful features
order_products_prior = pd.read_csv("~/jr_data/jr_project/imba_data_upload/order_products/order_products__prior.csv.gz")

In [50]:
# show df information
order_products_prior.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
dtypes: int64(4)
memory usage: 989.8 MB


In [33]:
# load orders into dataframe (ignore now order_dow, order_hour_of_day, days_since_prior)
orders = pd.read_csv("~/jr_data/jr_project/imba_data_upload/orders/orders.csv", usecols=["order_id","user_id","order_number"])

In [24]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 3 columns):
 #   Column        Dtype
---  ------        -----
 0   order_id      int64
 1   user_id       int64
 2   order_number  int64
dtypes: int64(3)
memory usage: 78.3 MB


In [51]:
# merge order_products_prior and orders (Q1 in assignment)
prior_order = pd.merge(order_products_prior, orders, how="inner", on="order_id")

In [52]:
# show df info
prior_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 6 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
 4   user_id            int64
 5   order_number       int64
dtypes: int64(6)
memory usage: 1.4 GB


In [27]:
del orders, order_products_prior

In [53]:
prior_order = prior_order.sort_values(by=['user_id', 'product_id', 'order_id'], ascending=[True, True, True]).reset_index()

In [54]:
prior_order.head(20)

,index,order_id,product_id,add_to_cart_order,reordered,user_id,order_number
0,4089398,431534,196,1,1,1,5
1,4488095,473747,196,1,1,1,3
2,5212927,550135,196,1,1,1,7
3,21376074,2254736,196,1,1,1,4
4,21760446,2295261,196,4,1,1,9
5,22742744,2398795,196,1,1,1,2
6,24076664,2539329,196,1,0,1,1
7,24181266,2550362,196,1,1,1,10
8,29474806,3108588,196,2,1,1,8
9,31927070,3367565,196,1,1,1,6


### Add features to the training data set (lastest orders) ###

In [34]:
# load order_product_train into dataframe to studies useful features
order_products_train = pd.read_csv("~/jr_data/jr_project/imba_data_upload/order_products/order_products__train.csv.gz")

# Merge order_products_train and orders (Q1 in sql assignment) 
# We use inner join to eliminate the rows in order_products_train that does not have previous order.
train_order = pd.merge(order_products_train, orders, how="inner", on="order_id")


In [35]:
train_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 6 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   order_id           1384617 non-null  int64
 1   product_id         1384617 non-null  int64
 2   add_to_cart_order  1384617 non-null  int64
 3   reordered          1384617 non-null  int64
 4   user_id            1384617 non-null  int64
 5   order_number       1384617 non-null  int64
dtypes: int64(6)
memory usage: 63.4 MB


In [36]:
# delete a few columns
train_order = train_order[["user_id", "product_id", "reordered"]]
train_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 3 columns):
 #   Column      Non-Null Count    Dtype
---  ------      --------------    -----
 0   user_id     1384617 non-null  int64
 1   product_id  1384617 non-null  int64
 2   reordered   1384617 non-null  int64
dtypes: int64(3)
memory usage: 31.7 MB


In [48]:
train_order.head(10)

,user_id,product_id,reordered
0,112108,49302,1
1,112108,11109,1
2,112108,10246,0
3,112108,49683,0
4,112108,43633,1
5,112108,13176,0
6,112108,47209,0
7,112108,22035,1
8,79431,39612,0
9,79431,19660,1


In [57]:
# Add features: for each pair of user_id and product_id, calculate the total number of orders, and sum of reordered
up_feature = prior_order.groupby(["user_id","product_id"])["reordered"].aggregate(["count", "sum"]).reset_index()

# Assign names to the columns in the up_feature
up_feature.columns = ["user_id", "product_id", "user_prd_count", "user_prd_sum"]

# Merge (left join) to show the main features (need to explain why left join)
train_order_up = pd.merge(train_order, up_feature, how="left", on=["user_id","product_id"])

In [64]:
train_order_up = pd.merge(train_order, up_feature, how="left", on=["user_id","product_id"])
train_order_up.sort_values(by=['user_id', 'product_id'], ascending=[True, True]).reset_index().head(20)

,index,user_id,product_id,reordered,user_prd_count,user_prd_sum
0,484420,1,196,1,10.0,9.0
1,484425,1,10258,1,9.0,8.0
2,484426,1,13032,1,3.0,2.0
3,484421,1,25133,1,8.0,7.0
4,484427,1,26088,1,2.0,1.0
5,484423,1,26405,1,2.0,1.0
6,484428,1,27845,0,NaN,NaN
7,484422,1,38928,1,1.0,0.0
8,484424,1,39657,1,1.0,0.0
9,484430,1,46149,1,3.0,2.0
